# Train the fretlift transcription model on Colab or Kaggle

**On Colab:**
1. **Runtime -> Change runtime type -> GPU** (a T4 is fine).
2. Run the cells in order. Datasets download to the machine's own disk,
   which is fast and doesn't use your Drive space. Runs (checkpoints, history)
   are saved to Google Drive.

**If the session disconnects:** run the Setup cells and the data cell again,
then re-run the training cell that was running. It picks up from its last
finished epoch.

**On Kaggle** (when Colab won't give you a GPU; Kaggle gives about 30 GPU hours a week):
1. In the notebook's settings, set **Accelerator -> GPU T4 x2** and turn
   **Internet** on (this needs a phone-verified Kaggle account).
2. Kaggle can't see your Google Drive. To carry on from Colab, download the
   `runs` and `reports` folders from Drive's `fretlift` folder, upload them as
   a private Kaggle dataset, and add it to the notebook (**Add Input**).
   Setup copies them in.
3. Click **Save Version -> Save & Run All**. It runs in the background, so you
   can close the browser. Runs that already finished are skipped, and a run
   with a `last.pt` carries on from it.
4. The whole version has to finish within 12 hours, or nothing is saved.
5. When it's done, download `fretlift/runs/...` from the version's **Output**
   tab. To continue on Colab, put those run folders in Drive's `fretlift/runs`.
   To continue on Kaggle, add the version's output as an input instead.

See `docs/TRAINING.md` in the repo for what each step does.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup

In [ ]:
import glob, os, shutil
KAGGLE = os.path.exists("/kaggle/input")
if KAGGLE:
    TMP = "/kaggle/temp"                      # this session only: datasets, the cache, the code
    WORK = "/kaggle/working/fretlift"         # runs and reports: saved as the version's output
else:
    from google.colab import drive
    drive.mount("/content/drive")
    TMP = "/content"                          # this session only: datasets, the cache, the code
    WORK = "/content/drive/MyDrive/fretlift"  # runs and reports: small, and they survive disconnects
DATA = f"{TMP}/data"
os.makedirs(f"{WORK}/reports", exist_ok=True)
if KAGGLE:  # runs and reports from added inputs: a dataset uploaded from Drive, or an earlier version's output
    for d in {os.path.dirname(f) for f in glob.glob("/kaggle/input/**/*.pt", recursive=True)}:
        if not os.path.exists(f"{WORK}/runs/{os.path.basename(d)}"):
            shutil.copytree(d, f"{WORK}/runs/{os.path.basename(d)}")
    for f in glob.glob("/kaggle/input/**/bp-*.json", recursive=True):
        shutil.copy(f, f"{WORK}/reports")
COMMON = f"--manifest {DATA}/guitarset/manifest.jsonl --cache-dir {TMP}/cache"
SPLIT = "--val-groups guitarset-player-04 --test-groups guitarset-player-05"
WORKERS = os.cpu_count()  # data loader processes: 2 on Colab, 4 on Kaggle

def resume(run):  # continue a run from its last finished epoch, if it has one
    last = f"{WORK}/runs/{run}/last.pt"
    return f"--out {WORK}/runs/{run}" + (f" --resume {last}" if os.path.exists(last) else "")

def done(run):  # finished runs are skipped, since Kaggle's Save & Run All runs every cell
    return os.path.exists(f"{WORK}/runs/{run}/test_report.json")

In [ ]:
REPO = "https://github.com/jdinkha/fretlift.git"
!git clone -q $REPO {TMP}/fretlift || (cd {TMP}/fretlift && git pull -q)
%cd {TMP}/fretlift
# Colab and Kaggle already have a CUDA build of PyTorch, so this only adds what's missing.
!pip install -q -e ".[train,dev]"
!pytest -q

## Step 1: data and the Basic Pitch baseline

GuitarSet is about 1.5 GB and Guitar-TECHS about 4.1 GB; each takes a minute
or two to download here. The baseline needs Basic Pitch; if installing it
conflicts with Colab's packages, skip the baseline cell here and run it in a
local virtual environment instead.

In [ ]:
!fretlift-data guitarset --root {DATA}/guitarset --download
!fretlift-data guitar-techs --root {DATA}/guitar-techs --download
!fretlift-data stats --manifest {DATA}/guitarset/manifest.jsonl --manifest {DATA}/guitar-techs/manifest.jsonl

In [ ]:
if not os.path.exists(f"{WORK}/reports/bp-sweep.json"):  # skipped once it has run
    !pip install -q basic-pitch
    !fretlift-eval --basic-pitch --split test --test-groups guitarset-player-05 {COMMON} --out {WORK}/reports/bp-clean.json
    !fretlift-eval --basic-pitch --split test --test-groups guitarset-player-05 {COMMON} --distort high_gain --out {WORK}/reports/bp-high_gain.json
    !fretlift-sweep --basic-pitch --split val --val-groups guitarset-player-04 {COMMON} --out {WORK}/reports/bp-sweep.json

## Step 3: train on clean audio

In [ ]:
if not done("gs-clean"):
    !fretlift-train {COMMON} {SPLIT} --workers {WORKERS} {resume("gs-clean")}

## Step 4: train with augmentation

Optional: upload cabinet impulse responses (WAV) to `$WORK/irs` and add
`--ir-dir {WORK}/irs`. Colab has 2 CPU cores (Kaggle 4), so augmented epochs are slower
(the CPU makes the audio, the GPU waits).

In [ ]:
if not done("gs-aug"):
    !fretlift-train {COMMON} {SPLIT} --workers {WORKERS} --augment full --val-distort high_gain {resume("gs-aug")}

## Step 5: add Guitar-TECHS (electric), starting from gs-aug

Starts from the gs-aug weights and writes a new run, so `runs/gs-aug` stays as
it is. Guitar-TECHS player 3 (the only real pieces of music) is held out for
validation, so the best checkpoint is picked on electric guitar too.

In [ ]:
TECHS = f"--manifest {DATA}/guitar-techs/manifest.jsonl"
VAL = "--val-groups guitarset-player-04,guitar-techs-P3 --test-groups guitarset-player-05"
if not done("gs-techs"):
    !fretlift-train {COMMON} {TECHS} {VAL} --workers {WORKERS} --augment full --val-distort high_gain --init {WORK}/runs/gs-aug/best.pt {resume("gs-techs")}

## Compare

In [ ]:
import json, pathlib
for run in ["gs-clean", "gs-aug", "gs-techs"]:
    p = pathlib.Path(WORK) / "runs" / run / "test_report.json"
    if p.exists():
        r = json.loads(p.read_text())
        for dist in [k for k in r if k != "decode"]:
            s = r[dist]["all"]
            print(f"{run:10s} {dist:10s} note F1 {s['note_f1']:.3f}  tab F1 {s['tab_f1'] or 0:.3f}")
for name in ["bp-clean", "bp-high_gain"]:
    p = pathlib.Path(WORK) / "reports" / f"{name}.json"
    if p.exists():
        s = json.loads(p.read_text())["summary"]["all"]
        print(f"{name:21s} note F1 {s['note_f1']:.3f}  tab F1 {s['tab_f1'] or 0:.3f}")

In [ ]:
import pandas as pd
pd.read_csv(f"{WORK}/runs/gs-techs/history.csv").plot(x="epoch", y=["clean_note_f1", "high_gain_note_f1", "clean_tab_f1"], grid=True)